In [ ]:
import sys; from pathlib import Path
src_dir = next((parent / 'src' for parent in Path().absolute().parents if (parent / 'src').is_dir()), None)
sys.path.append(str(src_dir))
from imports import *
from TASEP_models import *
importlib.reload(mi)
importlib.reload(ML)

GFP_TAG = 'GMDELYK' #'TYA'
HA_TAG = 'YPYDVPDYA'
U_TAG = 'MSLPGRWKPKM'
SUN_TAG = ''
MOON_TAG = ''

TAGS = [HA_TAG, GFP_TAG]
#TAGS = [U_TAG]

In [2]:
# # Initial conditions
ki = 0.04  # Initiation rate
global_elongation_rate = 10  # Elongation rates for positions 1 to N-1
number_repetitions = 200
folding_delay = 30
burnin_time = 500
timePerturbationApplication = 0 #5*60
t_max = 1000 #timePerturbationApplication + 25*60  # Maximum time
inhibitor_effectiveness=0.2
evaluatingInhibitor = 0


In [ ]:
gfp_6_copies = True
if gfp_6_copies == True:
    file_path = pathlib.Path('pNZ212(pUB-mRuby2HA-6xsfGFP-24xMS2).dna')
else:
    file_path = pathlib.Path('pNZ251_pUB-mRuby2HA-1xsfGFP-24xMS2.dna')

#file_path = pathlib.Path('pNZ208(pUB-24xUTagFullLength-KDM5B-MS2).dna')

# reading the sequence and extracting the elongation rates
protein, rna, dna, indexes_tags, _, seq_record, graphic_features  = read_sequence(seq=file_path, min_protein_length=50,TAG=TAGS)
plasmid_figure = plot_plasmid(seq_record, graphic_features,figure_width=25, figure_height=3)

gene_length = len(protein)+1 # adding 1 to account for the stop codon
tag_positions_first_probe_vector = indexes_tags[0]
tag_positions_second_probe_vector = indexes_tags[1] if len(indexes_tags) > 1 else None

first_probe_position_vector = create_probe_vector(tag_positions_first_probe_vector, gene_length)
second_probe_position_vector = create_probe_vector(tag_positions_second_probe_vector, gene_length) if tag_positions_second_probe_vector is not None else None


In [ ]:
file_path.name.split('.')[0]
plasmid_name = file_path.name.split('.')[0].replace('(','_').replace(')','_')
plasmid_name

In [5]:
ke = calculate_codon_elongation_rates (rna, global_elongation_rate=global_elongation_rate)

In [6]:
use_pause = False
if use_pause:
    # adding a pause site by setting the elongation rate to 0.001
    ke[-5] = 1/(global_elongation_rate+5) # 1/60 is the elongation rate of the pause site. Meaning this codon takes 60 seconds to be translated. 

## Deterministic modeling
____

In [7]:
intensity_vector_first_signal_ode,intensity_vector_second_signal_ode = simulate_TASEP_ODE(ki, ke, gene_length, t_max,first_probe_position_vector,second_probe_position_vector,burnin_time)
# plt.plot(intensity_vector_first_signal_ode/np.max(intensity_vector_first_signal_ode))
# plt.plot(intensity_vector_second_signal_ode/np.max(intensity_vector_second_signal_ode))
# plt.show()


# Modeling TASEP SSA
____

In [ ]:
list_ribosome_trajectories, list_occupancy_output, matrix_intensity_first_signal_RT, matrix_intensity_second_signal_RT = simulate_TASEP_SSA(ki, ke, gene_length, t_max,number_repetitions, first_probe_position_vector,second_probe_position_vector,folding_delay=folding_delay,timePerturbationApplication=timePerturbationApplication, evaluatingInhibitor=evaluatingInhibitor,burnin_time=burnin_time,inhibitor_effectiveness=inhibitor_effectiveness)

In [ ]:
# calculate the ribosomal occupancy
theoretical_occupancy = (gene_length/global_elongation_rate) *ki
theoretical_occupancy

In [10]:
# calculate the mean and std of the matrix_intensity_first_signal_RT and matrix_intensity_second_signal_RT
mean_first_signal_RT = np.mean(matrix_intensity_first_signal_RT, axis=0)
sem_first_signal_RT = np.std(matrix_intensity_first_signal_RT, axis=0)/np.sqrt(number_repetitions)
if second_probe_position_vector is not None:
    mean_second_signal_RT = np.mean(matrix_intensity_second_signal_RT, axis=0)
    sem_second_signal_RT = np.std(matrix_intensity_second_signal_RT, axis=0)/np.sqrt(number_repetitions)

In [ ]:
# plot a single trajectory for the the two signals
plt.figure()
selected_trajectory = 0
plt.plot(matrix_intensity_first_signal_RT[selected_trajectory,:]/np.max(matrix_intensity_first_signal_RT[selected_trajectory,:]), label='first signal')
if second_probe_position_vector is not None:
    plt.plot(matrix_intensity_second_signal_RT[selected_trajectory,:]/np.max(matrix_intensity_second_signal_RT[selected_trajectory,:]), label='second signal')
plt.legend()
plt.show()

In [ ]:
# plot the mean and std as error shade
downsample = 50
downsampled_time = np.arange(0,t_max,downsample)

plt.figure(figsize=(5,4))
plt.plot(mean_first_signal_RT,color = 'k',linewidth=2, label='SSA')
plt.fill_between(np.arange(len(mean_first_signal_RT)), mean_first_signal_RT-sem_first_signal_RT, mean_first_signal_RT+sem_first_signal_RT, color='k', alpha=0.2)
plt.plot(downsampled_time, intensity_vector_first_signal_ode[::downsample], color = 'blue', linestyle='dashed', marker='o', label='ODE')
if second_probe_position_vector is not None:
    # plot for the second signal
    plt.plot(mean_second_signal_RT,color = 'k',linewidth=2, label='SSA')
    plt.fill_between(np.arange(len(mean_second_signal_RT)), mean_second_signal_RT-sem_second_signal_RT, mean_second_signal_RT+sem_second_signal_RT, color='k', alpha=0.2)
    plt.plot(downsampled_time, intensity_vector_second_signal_ode[::downsample], color = 'red', linestyle='dashed', marker='o', label='ODE')

plt.xlabel('Time')
plt.ylabel('Intensity')
plt.legend()
plt.show()

# Plot ribosome movement
___

In [13]:
selected_trajectory = 0

#list_ribosome_trajectories, list_occupancy_output, matrix_intensity_first_signa_RT, matrix_intensity_second_signa_RT 
ribosome_trajectories = list_ribosome_trajectories[selected_trajectory]    
ribosome_trajectories = ribosome_trajectories[:,:]
intensity_vector_first_signal = matrix_intensity_first_signal_RT[selected_trajectory,:]
if second_probe_position_vector is not None:
    intensity_vector_second_signal = matrix_intensity_second_signal_RT[selected_trajectory,:]
else:
    intensity_vector_second_signal = None
#plot_RibosomeMovement(ribosome_trajectories, intensity_vector_first_signal ,tag_positions_first_probe_vector,SecondIntensityVector=intensity_vector_second_signal,second_probePositions=tag_positions_second_probe_vector,timePerturbationApplication=timePerturbationApplication) # intensity_vector_second_signal

In [ ]:
ribosome_trajectories.shape
# make a binary matrix of the ribosome trajectories that are more than 0
binary_matrix_ribosomal_occupancy = ribosome_trajectories > 0
binary_matrix_ribosomal_occupancy.shape
# sum the binary matrix to get the ribosomal occupancy at each time point
ribosomal_occupancy = np.sum(binary_matrix_ribosomal_occupancy, axis=0)
# print mean ribosomal occupancy
print( 'mean ribosomal occupancy: ', np.mean(ribosomal_occupancy) )



In [ ]:
str_ki = str(ki).replace('.','_')
str_k = str(global_elongation_rate).replace('.','_')
fileNameGif = 'simulation_'+plasmid_name+'_ke_'+str_k+'_ki_'+str_ki + '_inhibitor_effectiveness_'+str(inhibitor_effectiveness)
plot_RibosomeMovement_and_Microscope(ribosome_trajectories, intensity_vector_first_signal, tag_positions_first_probe_vector, SecondIntensityVector=intensity_vector_second_signal, second_probePositions=tag_positions_second_probe_vector,FrameVelocity=20,timePerturbationApplication=timePerturbationApplication,fileNameGif=fileNameGif)

In [ ]:
downsample_time = 1
downsample_replicates = 1
burnin = 200



matrix_intensity_first_signal_RT_downsampled = matrix_intensity_first_signal_RT[:,burnin:][::downsample_replicates,::downsample_time]
matrix_intensity_second_signal_RT_downsampled = matrix_intensity_second_signal_RT[:,burnin:][::downsample_replicates,::downsample_time]

print(matrix_intensity_second_signal_RT_downsampled.shape)

In [17]:
# normalize the data to be between 0 and 255
# normalize to the 99 percentile and then multiply by 255 and clip to 255
matrix_intensity_first_signal_RT_downsampled_uint8 = (matrix_intensity_first_signal_RT_downsampled/np.percentile(matrix_intensity_first_signal_RT_downsampled,99)*255).clip(0,255).astype(int)
matrix_intensity_second_signal_RT_downsampled_uint8 = (matrix_intensity_second_signal_RT_downsampled/np.percentile(matrix_intensity_second_signal_RT_downsampled,99)*255).clip(0,255).astype(int)


#matrix_intensity_first_signal_RT_downsampled_uint8 = (matrix_intensity_first_signal_RT_downsampled/np.max(matrix_intensity_first_signal_RT_downsampled)*255).astype(int)
#matrix_intensity_second_signal_RT_downsampled_uint8 = (matrix_intensity_second_signal_RT_downsampled/np.max(matrix_intensity_second_signal_RT_downsampled)*255).astype(int)

In [ ]:
matrix_intensity_first_signal_RT_downsampled_uint8.shape

In [19]:
import numpy as np
import matplotlib.pyplot as plt

def gaussian_2d(x, y, x0, y0, amplitude, sigma):
    """Generate values for a 2D Gaussian."""
    return amplitude * np.exp(-((x - x0)**2 + (y - y0)**2) / (2 * sigma**2))

# User-defined parameters
radius = 200  # Radius of the cell in pixels
center_x, center_y = 256, 256  # Center of the circle

# Check if a point is inside the circle
def is_inside_circle(x, y, center_x, center_y, radius):
    return (x - center_x)**2 + (y - center_y)**2 <= radius**2


n = matrix_intensity_first_signal_RT_downsampled_uint8.shape[0]  # Number of Gaussian spots (particles)
m = matrix_intensity_first_signal_RT_downsampled_uint8.shape[1]  # Number of frames
#amplitude = 255  # Amplitude of the Gaussian spots
sigma = 2.12  # Sigma of the Gaussian (controls the width)
diffusion_coefficient = 0.01  # Diffusion coefficient

np.random.seed(0)
positions = np.empty((n, 2))

# Initialize positions within the circle
for i in range(n):
    while True:
        x = np.random.uniform(center_x - radius, center_x + radius)
        y = np.random.uniform(center_y - radius, center_y + radius)
        if is_inside_circle(x, y, center_x, center_y, radius):
            positions[i] = [x, y]
            break

# Create a grid of (x, y) coordinates for the image
x = np.linspace(0, 511, 512)
y = np.linspace(0, 511, 512)
x, y = np.meshgrid(x, y)

frames_first_channel = []
frames_second_channel = []

# Simulate movement over m frames
for frame in range(m):
    background = np.zeros((512, 512))  # Reset background for each frame
    background_second_channel = np.zeros((512, 512))  # Reset background for each frame
    # Update positions via random walk
    steps = np.random.normal(0, np.sqrt(diffusion_coefficient), (n, 2))
    new_positions = positions + steps
    # Ensure particles remain inside the circle
    for i in range(n):
        if not is_inside_circle(new_positions[i, 0], new_positions[i, 1], center_x, center_y, radius):
            while True:
                steps = np.random.normal(0, np.sqrt(diffusion_coefficient), (1, 2))
                new_positions[i] = positions[i] + steps
                if is_inside_circle(new_positions[i, 0], new_positions[i, 1], center_x, center_y, radius):
                    break
    positions = new_positions
    # Draw each Gaussian spot within the circle
    #for pos in positions:
    for index_particle, pos in enumerate( positions):
        background += gaussian_2d(x, y, int(pos[0]), int(pos[1]), matrix_intensity_first_signal_RT_downsampled_uint8[index_particle,frame], sigma)
        background_second_channel += gaussian_2d(x, y, int(pos[0]), int(pos[1]), matrix_intensity_second_signal_RT_downsampled_uint8[index_particle,frame], sigma)
    frames_first_channel.append(background)
    frames_second_channel.append(background_second_channel)


In [ ]:
# convert the frames to a numpy array
frames_first_channel_array = np.array(frames_first_channel)
frames_second_channel_array = np.array(frames_second_channel)
# create a new dimension for the color channel
frames_first_channel_exp = np.expand_dims(frames_first_channel_array, axis=-1)
frames_second_channel_exp = np.expand_dims(frames_second_channel_array, axis=-1)
# concatenate the two channels
simulated_cell = np.concatenate((frames_first_channel_exp, frames_second_channel_exp), axis=-1)
simulated_cell.shape

In [ ]:
# plot the simulated cell, the first and the last frames and the first and second color channels
plt.figure(figsize=(10,7))
plt.subplot(231)
plt.imshow(simulated_cell[0,:,:,0], cmap='winter_r')
plt.title('First frame, first channel')
plt.subplot(232)
plt.imshow(simulated_cell[100,:,:,0], cmap='winter_r')
plt.title('First frame, second channel')
plt.subplot(233)
plt.imshow(simulated_cell[-1,:,:,0], cmap='winter_r')
plt.title('First frame, both channels')
plt.subplot(234)
plt.imshow(simulated_cell[0,:,:,1], cmap='hot')
plt.title('Last frame, first channel')
plt.subplot(235)
plt.imshow(simulated_cell[100,:,:,1], cmap='hot')
plt.title('Last frame, second channel')
plt.subplot(236)
plt.imshow(simulated_cell[-1,:,:,1], cmap='hot')
plt.title('Last frame, both channels')

plt.show()


In [ ]:
simulated_cell.shape

In [ ]:
def section_image(image_TYXC, number_rows=25, number_cols=25):
    # Dimensions of each subsection
    number_time_points = image_TYXC.shape[0]
    sub_width = image_TYXC.shape[2] // number_cols
    sub_height = image_TYXC.shape[1] // number_rows
    number_color_channels = image_TYXC.shape[3]
    
    # Initialize the numpy array to store the subsections' intensity data
    intensity_subsections_matrix_Section_Time_Colors = np.zeros(
        (number_rows * number_cols, number_time_points, number_color_channels)
    )
    
    # Iterate through each time point and channel
    for t in range(number_time_points):
        for c in range(number_color_channels):
            for i in range(number_rows):
                for j in range(number_cols):
                    # Calculate the boundaries of the subsection
                    start_x = j * sub_width
                    end_x = start_x + sub_width
                    start_y = i * sub_height
                    end_y = start_y + sub_height

                    # Extract the subsection for the specific time point and channel
                    subsection = image_TYXC[t, start_y:end_y, start_x:end_x, c]
                    # Calculate the total intensity of the subsection
                    total_intensity = np.sum(subsection)
                    # Calculate the index for the current subsection
                    index = i * number_cols + j

                    # Save the intensity in the matrix
                    intensity_subsections_matrix_Section_Time_Colors[index, t, c] = total_intensity

    return intensity_subsections_matrix_Section_Time_Colors

# Create subsections of the simulated cell  
intensity_subsections_matrix_Section_Time_Colors = section_image(simulated_cell, number_rows=30, number_cols=30)
intensity_subsections_matrix_Section_Time_Colors.shape


In [ ]:
# Plot the intensity of the subsections over time for the first color channel
plt.figure(figsize=(10, 5))
plt.plot(intensity_subsections_matrix_Section_Time_Colors[:, :, 0].T, color='blue', alpha=0.1)
plt.xlabel('Time')
plt.ylabel('Intensity')
plt.title('Intensity of Subsections Over Time (First Color Channel)')
plt.show()

# Plot the intensity of the subsections over time for the second color channel

plt.figure(figsize=(10, 5))
plt.plot(intensity_subsections_matrix_Section_Time_Colors[:, :, 1].T, color='red', alpha=0.1)
plt.xlabel('Time')
plt.ylabel('Intensity')
plt.title('Intensity of Subsections Over Time (Second Color Channel)')
plt.show()

In [ ]:
# remove rows in intensity_time_series_matrix with mean less than 1000
mean_intensity = np.mean(intensity_subsections_matrix_Section_Time_Colors[:, :, 0].T, axis=0)
# remove the 50% of the data with the lowest mean intensity
mask = mean_intensity > np.mean(mean_intensity)*6
# remove the rows from all color channels that have a mean intensity less than 1000
#mask = mean_intensity > 2000
intensity_subsections_matrix_Section_Time_Colors_filtered = intensity_subsections_matrix_Section_Time_Colors[mask,:,:]
intensity_subsections_matrix_Section_Time_Colors_filtered.shape


In [ ]:
# Plot the intensity of the subsections over time for the first color channel
plt.figure(figsize=(10, 5))
plt.plot(intensity_subsections_matrix_Section_Time_Colors_filtered[:, :, 0].T, color='blue', alpha=0.1)
plt.xlabel('Time')
plt.ylabel('Intensity')
plt.title('Intensity of Subsections Over Time (First Color Channel)')
plt.show()

# Plot the intensity of the subsections over time for the second color channel

plt.figure(figsize=(10, 5))
plt.plot(intensity_subsections_matrix_Section_Time_Colors_filtered[:, :, 1].T, color='red', alpha=0.1)
plt.xlabel('Time')
plt.ylabel('Intensity')
plt.title('Intensity of Subsections Over Time (Second Color Channel)')
plt.show()

In [ ]:
mean_correlation_ch0_boxing, std_correlation_ch0, lags_ch0, correlations_array_ch0, dwell_time_ch0 = mi.Correlation(primary_data=intensity_subsections_matrix_Section_Time_Colors_filtered[...,0], max_lag=None, nan_handling='zeros',shift_data=False,return_full=False,time_interval_between_frames_in_seconds=1,show_plot=True,start_lag=0,fit_type='linear',de_correlation_threshold=0.001).run()


In [ ]:
def exponential(x, a, b):
    return a * np.exp(-b * x)

# initial guess
p0 = (mean_correlation_ch0_boxing[0], 1)
popt, pcov = curve_fit(exponential, lags_ch0, mean_correlation_ch0_boxing, p0=p0)

plt.plot(lags_ch0, mean_correlation_ch0_boxing, 'b-', label='data')
plt.plot(lags_ch0, exponential(lags_ch0, *popt), 'r-', label='fit: a=%5.3f, b=%5.3f' % tuple(popt))
plt.show()

# print the fit parameters
dwell_time = np.log(2)/popt[1]


full_decorrelation_time = np.log(0.001) / -popt[1]
print(f"Full Decorrelation Time (1% of initial value): {full_decorrelation_time:.3f} time units")

# calculate the elongation rate
ke_calculated =  np.round( (gene_length) /full_decorrelation_time  , 2)
print(ke_calculated)

# initiation rate
ki_calculated = 1/ (mean_correlation_ch0_boxing[0] * full_decorrelation_time)
print(ki_calculated)


In [ ]:
mean_correlation_c1_boxing, std_correlation_ch1, lags_ch1, correlations_array_ch1, dwell_time_ch1 = mi.Correlation(primary_data=intensity_subsections_matrix_Section_Time_Colors_filtered[...,1], max_lag=None, nan_handling='zeros',shift_data=False,return_full=False,time_interval_between_frames_in_seconds=1,show_plot=True,start_lag=1,fit_type='exponential',de_correlation_threshold=0.001).run()


In [ ]:
def exponential(x, a, b):
    return a * np.exp(-b * x)

# initial guess
p0 = (mean_correlation_c1_boxing[0], 1)
popt, pcov = curve_fit(exponential, lags_ch0, mean_correlation_c1_boxing, p0=p0)

plt.plot(lags_ch0, mean_correlation_c1_boxing, 'b-', label='data')
plt.plot(lags_ch0, exponential(lags_ch0, *popt), 'r-', label='fit: a=%5.3f, b=%5.3f' % tuple(popt))
plt.show()

# print the fit parameters
dwell_time = np.log(2)/popt[1]


full_decorrelation_time = np.log(0.001) / -popt[1]
print(f"Full Decorrelation Time (1% of initial value): {full_decorrelation_time:.3f} time units")

# calculate the elongation rate
ke_calculated =  np.round( (gene_length) /full_decorrelation_time  , 2)
print(ke_calculated)

# initiation rate
ki_calculated = 1/ (mean_correlation_c1_boxing[0] * full_decorrelation_time)
print(ki_calculated)


In [ ]:
intensity_subsections_matrix_Section_Time_Colors_filtered[...,0].shape

In [ ]:
mean_cross_correlation, std_cross_correlation, lags_cross_correlation, cross_correlations_array, max_lag = mi.Correlation(primary_data=intensity_subsections_matrix_Section_Time_Colors_filtered[...,0], secondary_data=intensity_subsections_matrix_Section_Time_Colors_filtered[...,1], max_lag=None, nan_handling='zeros', shift_data=False, return_full=True,time_interval_between_frames_in_seconds=1,show_plot=True).run()

In [ ]:
raise Exception('stop here')

In [ ]:
#mean_correlation_ch0_boxing, std_correlation_ch0, lags_ch0, correlations_array_ch0, dwell_time_ch0 = mi.Correlation(primary_data=intensity_subsections_matrix_Section_Time_Colors_filtered[...,0], max_lag=None, nan_handling='zeros',shift_data=False,return_full=False,time_interval_between_frames_in_seconds=1,show_plot=True,start_lag=1,fit_type='exponential',de_correlation_threshold=0.001).run()


In [ ]:
intensity_time_series_matrix = np.array(intensity_time_series)  
intensity_time_series_matrix.shape

In [ ]:
# remove rows in intensity_time_series_matrix with mean less than 1000
mean_intensity = np.mean(intensity_time_series_matrix, axis=1)
std_intensity = np.std(intensity_time_series_matrix, axis=1)

# remove the 50% of the data with the lowest mean intensity
mask = mean_intensity > np.mean(mean_intensity)*2
intensity_time_series_matrix_filtered = intensity_time_series_matrix[mask,:]
intensity_time_series_matrix_filtered.shape

In [ ]:
mean_correlation_ch0_boxing, std_correlation_ch0, lags_ch0, correlations_array_ch0, dwell_time_ch0 = mi.Correlation(primary_data=intensity_time_series_matrix_filtered, max_lag=None, nan_handling='zeros',shift_data=False,return_full=False,time_interval_between_frames_in_seconds=1,show_plot=True,start_lag=1,fit_type='exponential',de_correlation_threshold=0.001).run()
#print(gene_length/dwell_time_ch0)
#ke_calculated_ch0 =  np.round( (gene_length-np.max(tag_positions_first_probe_vector)) /dwell_time_ch0  , 2)
#print(ke_calculated_ch0)

In [ ]:


# fit the data to an exponential function
def exponential(x, a, b):
    return a * np.exp(-b * x)

# initial guess
p0 = (mean_correlation_ch0_boxing[0], 1)
popt, pcov = curve_fit(exponential, lags_ch0, mean_correlation_ch0_boxing, p0=p0)

plt.plot(lags_ch0, mean_correlation_ch0_boxing, 'b-', label='data')
plt.plot(lags_ch0, exponential(lags_ch0, *popt), 'r-', label='fit: a=%5.3f, b=%5.3f' % tuple(popt))
plt.show()

# print the fit parameters
dwell_time = np.log(2)/popt[1]

print(dwell_time)

full_decorrelation_time = np.log(0.001) / -popt[1]
print(f"Full Decorrelation Time (1% of initial value): {full_decorrelation_time:.3f} time units")

# calculate the elongation rate
ke_calculated =  np.round( (gene_length) /full_decorrelation_time  , 2)
print(ke_calculated)


# initiation rate
ki_calculated = 1/ (mean_correlation_ch0_boxing[0] * full_decorrelation_time)
print(ki_calculated)


In [ ]:
# initiation rate
ki_calculated = 1/ (mean_correlation_ch0_boxing[0] * full_decorrelation_time)
print(ki_calculated)

In [ ]:
ke_calculated_ch0 =  np.round( (gene_length-np.max(tag_positions_first_probe_vector)/2) /full_decorrelation_time  , 2)
ke_calculated_ch0


In [32]:
# convert the frames to a gif
import imageio
imageio.mimsave('diffusion_simulation.gif', frames, duration=0.1)


## Calculating Correlations
____

In [33]:
def find_last_valid_column(data):
    # Initialize an array to hold the index of the last valid data point for each row
    last_valid_indices = np.zeros(data.shape[0], dtype=int)
    # Process each row individually
    for idx, row in enumerate(data):
        # Reverse the row to make counting consecutive NaNs from the end easier
        reversed_row = np.flip(row)
        # Find the first non-NaN value in the reversed row
        valid_index = np.argmax(~np.isnan(reversed_row))
        # If the entire reversed row is NaN, the valid_index will point to a NaN
        if np.isnan(reversed_row[valid_index]):
            # If no valid data points are found, mark the index as -1 or another flag value
            last_valid_indices[idx] = -1
        else:
            # Calculate the last valid index in the original row
            last_valid_indices[idx] = len(row) - 1 - valid_index
    return np.max(last_valid_indices)

def shift_initial_nans(data):
    # Create a new array of the same shape filled with NaNs
    new_data = np.full(data.shape, np.nan)
    # Iterate over each row
    for idx, row in enumerate(data):
        # Find the index of the first non-NaN value
        first_non_nan_index = np.argmax(~np.isnan(row))
        # Check if the row has any non-NaNs at all
        if not np.isnan(row[first_non_nan_index]):
            # Number of elements to shift
            elements_to_shift = len(row) - first_non_nan_index
            # Shift the elements from the first non-NaN to the left
            new_data[idx, :elements_to_shift] = row[first_non_nan_index:]
    return new_data

def simulate_missing_data(matrix1, matrix2=None, percentage_to_remove_data=0, replace_with='zeros'):
    if percentage_to_remove_data ==0: 
        return matrix1, matrix2
    if matrix1.shape != matrix2.shape:
        raise ValueError("Both matrices must have the same shape.")
    num_rows, num_cols = matrix1.shape
    new_matrix1 = matrix1.copy()
    if matrix2 is not None:
        new_matrix2 = matrix2.copy()
    total_cols_to_remove = int(num_cols * (percentage_to_remove_data / 100))
    if total_cols_to_remove >= num_cols:
        raise ValueError("Percentage to remove too high, no columns left to keep.")
    # Determine replacement value (zero or NaN)
    if replace_with == 'zeros':
        replacement_value = 0
    elif replace_with == 'nan':
        replacement_value = np.nan
    else:
        raise ValueError("Invalid replace_with argument. Use 'zeros' or 'nan'.")
    for i in range(num_rows):
        # Randomly split the total columns to remove between left and right
        left_cols_to_remove = np.random.randint(0, total_cols_to_remove + 1)
        right_cols_to_remove = total_cols_to_remove - left_cols_to_remove
        # Replace the columns from the extremes in both matrices
        if left_cols_to_remove > 0:
            new_matrix1[i, :left_cols_to_remove] = replacement_value
            if matrix2 is not None:
                new_matrix2[i, :left_cols_to_remove] = replacement_value
        if right_cols_to_remove > 0:
            new_matrix1[i, num_cols - right_cols_to_remove:] = replacement_value
            if matrix2 is not None:
                new_matrix2[i, num_cols - right_cols_to_remove:] = replacement_value
    if matrix2 is None:
        return new_matrix1, None # Return only the first matrix if the second one is None
    else:   
        return new_matrix1, new_matrix2
    
def simulate_photobleaching(matrix, decay_rate):
    num_rows, num_cols = matrix.shape
    # Generate the time points (column indices) for the decay
    time_points = np.arange(num_cols)
    # Calculate the exponential decay factor for each time point
    decay_factors = np.exp(-decay_rate * time_points)
    # Apply the decay equally to each row
    decayed_matrix = matrix * decay_factors
    return decayed_matrix


def plot_intensity_trajectory_arrays_(array1, array2=None):
    """
    Plots one or two matrices. If the second matrix is None, only the first matrix is plotted.
    
    Args:
    array1 (np.ndarray): First matrix to plot.
    array2 (np.ndarray, optional): Second matrix to plot. Defaults to None.
    """
    # Determine the number of subplots
    if array2 is None:
        fig, ax = plt.subplots(figsize=(12, 4))
        ax.matshow(array1, aspect='auto', cmap='hot')
        ax.set_title('First Signal')
        ax.grid(False)
    else:
        fig, axs = plt.subplots(2, 1, figsize=(12, 6))
        # Plot the first array
        axs[0].matshow(array1, aspect='auto', cmap='hot')
        axs[0].set_title('First Signal')
        axs[0].grid(False)
        # Plot the second array
        axs[1].matshow(array2, aspect='auto', cmap='hot')
        axs[1].set_title('Second Signal')
        axs[1].grid(False)
    plt.tight_layout()  # Adjust layout to prevent overlap
    plt.show()


In [ ]:
importlib.reload(mi)
time_interval_between_frames_in_seconds = 1 # seconds
burnin = 500
downsample_time = 1
downsample_replicates = 1
percentage_to_remove_data = 50  # Remove 80% of the data
shift_data = False
simulate_photobleacing = False


matrix_intensity_first_signal_RT_downsampled = matrix_intensity_first_signal_RT[:,burnin:][::downsample_replicates,::downsample_time]
matrix_intensity_second_signal_RT_downsampled = matrix_intensity_second_signal_RT[:,burnin:][::downsample_replicates,::downsample_time]
print('number of replicates : ', matrix_intensity_first_signal_RT_downsampled.shape[0], '\nnumber of time points : ', matrix_intensity_first_signal_RT_downsampled.shape[1])

if simulate_photobleacing:
    decay_rate_first_signal = -np.log(0.8) / 100  # 20% decrease after 100 minutes
    decay_rate_second_signal = -np.log(0.8) / 100 # 10% decrease after 100 minutes
    matrix_intensity_first_signal_RT_downsampled = simulate_photobleaching(matrix_intensity_first_signal_RT_downsampled, decay_rate_first_signal)
    matrix_intensity_second_signal_RT_downsampled = simulate_photobleaching(matrix_intensity_second_signal_RT_downsampled, decay_rate_second_signal)
    plot_intensity_trajectory_arrays_(matrix_intensity_first_signal_RT_downsampled, matrix_intensity_second_signal_RT_downsampled)

In [ ]:
matrix_intensity_first_signal_RT_downsampled,matrix_intensity_second_signal_RT_downsampled = simulate_missing_data(matrix_intensity_first_signal_RT_downsampled,matrix_intensity_second_signal_RT_downsampled, percentage_to_remove_data,replace_with='nan')

if shift_data == True:
    matrix_intensity_first_signal_RT_downsampled = shift_initial_nans(matrix_intensity_first_signal_RT_downsampled)
    matrix_intensity_second_signal_RT_downsampled = shift_initial_nans(matrix_intensity_second_signal_RT_downsampled)
    last_valid_columns = find_last_valid_column(matrix_intensity_first_signal_RT_downsampled)
    matrix_intensity_first_signal_RT_downsampled = matrix_intensity_first_signal_RT_downsampled[:, :last_valid_columns]
    matrix_intensity_second_signal_RT_downsampled = matrix_intensity_second_signal_RT_downsampled[:, :last_valid_columns]
plot_intensity_trajectory_arrays_(matrix_intensity_first_signal_RT_downsampled, matrix_intensity_second_signal_RT_downsampled)


In [ ]:
mean_correlation_ch0, std_correlation_ch0, lags_ch0, correlations_array_ch0, dwell_time_ch0 = mi.Correlation(primary_data=matrix_intensity_first_signal_RT_downsampled, max_lag=None, nan_handling='zeros',shift_data=True,return_full=False,time_interval_between_frames_in_seconds=time_interval_between_frames_in_seconds*downsample_time,show_plot=True,start_lag=1,fit_type='linear',de_correlation_threshold=0.001).run()


In [ ]:
ke_calculated_ch0 =  np.round( (gene_length-np.max(tag_positions_first_probe_vector)/2) /dwell_time_ch0  , 2)
ke_calculated_ch0


In [ ]:
mean_correlation_ch1_tracking, std_correlation_ch1, lags_ch1, correlations_array_ch1, dwell_time_ch1 = mi.Correlation(primary_data=matrix_intensity_second_signal_RT_downsampled, max_lag=None, nan_handling='zeros',shift_data=True,return_full=False,time_interval_between_frames_in_seconds=time_interval_between_frames_in_seconds*downsample_time,show_plot=True,start_lag=1,fit_type='linear',de_correlation_threshold=0.001).run()

In [ ]:
ke_calculated_ch1 = np.round( abs( (gene_length-np.max(tag_positions_second_probe_vector)/(1/2))) /dwell_time_ch1  , 2)
ke_calculated_ch1

In [ ]:
# initiation rate
ki_calculated = 1/ (mean_correlation_ch1_tracking[0] * dwell_time_ch1)
print(ki_calculated)

In [ ]:
mean_cross_correlation, std_cross_correlation, lags_cross_correlation, cross_correlations_array, max_lag = mi.Correlation(primary_data=matrix_intensity_first_signal_RT_downsampled, secondary_data=matrix_intensity_second_signal_RT_downsampled, max_lag=None, nan_handling='zeros', shift_data=True, return_full=True,time_interval_between_frames_in_seconds=time_interval_between_frames_in_seconds*downsample_time,show_plot=True).run()

In [ ]:
mean_cross_correlation.shape

In [ ]:
plt.plot(mean_cross_correlation)
# get the max lag for the cross correlation
max_lag = np.argmax(mean_cross_correlation)
max_lag
# get the index of the maximum lag
max_lag_index = np.argmax(mean_cross_correlation)
max_lag_index

# set the xlim to 50 values before and after the max lag
range_lag = 250
plt.xlim(max_lag_index-range_lag,max_lag_index+range_lag) 
# plot an horizontal line at max_lag_index
plt.axvline(x=max_lag_index, color='k', linestyle='--')


In [ ]:
# robosmal occupancy
# calculate the ribosome occupancy


In [ ]:
raise ValueError('stop here')

## Testing the effects of incomplete data on the calculation of the elongation rates and time delays.
____ 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set the total time and time resolution
T_total = 5000  # Total time in seconds
dt = 0.1       # Time step in seconds
t = np.arange(0, T_total + dt, dt)  # Time vector

# Initialize the signals
signal_A = np.zeros_like(t)
signal_B = np.zeros_like(t)

# Random seed for reproducibility (optional)
np.random.seed(42)

# Parameters for the signals
N_steps_A = 10          # Number of steps for Signal A
step_interval_A = 0.1     # Time between steps in Signal A (seconds)
duration_A_steps = (N_steps_A - 1) * step_interval_A  # Duration of steps in Signal A

N_steps_B = 1          # Number of steps for Signal B
step_interval_B = 20    # Time between steps in Signal B (seconds)
duration_B = (N_steps_B - 1) * step_interval_B  # Total duration of Signal B

# Initiation rate parameter (occurrences per second)
initiation_rate = 0.01  # Adjust this value to set the initiation rate

# Calculate the expected number of occurrences
expected_occurrences = int(T_total * initiation_rate)

# Generate initiation times using exponential distribution (inter-arrival times)
inter_arrival_times = np.random.exponential(scale=1/initiation_rate, size=expected_occurrences)
start_times = np.cumsum(inter_arrival_times)

# Filter out start times beyond T_total
start_times = start_times[start_times < T_total - max(duration_A_steps, duration_B) - 1]  # Leave 1 second buffer

# Loop over initiation times
for start_time_A in start_times:
    # Build step times for Signal A
    step_times_A = start_time_A + np.arange(N_steps_A) * step_interval_A
    end_time_A_steps = step_times_A[-1]  # Time when Signal A reaches max amplitude
    
    # Random delay before Signal B starts after Signal A starts
    # Ensure Signal B starts after Signal A starts but before Signal A finishes stepping
    max_delay_B = end_time_A_steps - start_time_A
    if max_delay_B <= 0:
        continue  # Cannot start Signal B after Signal A steps are done
    delay_B_after_A = np.random.uniform(0, max_delay_B)
    start_time_B = start_time_A + delay_B_after_A
    
    # Build step times for Signal B
    step_times_B = start_time_B + np.arange(N_steps_B) * step_interval_B
    end_time_B = step_times_B[-1]  # Time when Signal B reaches max amplitude
    
    # Determine the time when both signals disappear (after Signal B reaches maximum)
    end_time_occurrence = end_time_B + dt  # Add dt to ensure signals disappear after last step
    
    # Build Signal A: steps until maximum amplitude, then maintain until end_time_occurrence
    amplitude_A = 0
    for step_time in step_times_A:
        idx = np.where((t >= step_time) & (t < end_time_occurrence))[0]
        if idx.size > 0:
            amplitude_A += 1
            signal_A[idx] = amplitude_A  # Maintain amplitude until signals disappear
    
    # Build Signal B: steps until maximum amplitude, then maintain until end_time_occurrence
    amplitude_B = 0
    for step_time in step_times_B:
        idx = np.where((t >= step_time) & (t < end_time_occurrence))[0]
        if idx.size > 0:
            amplitude_B += 1
            signal_B[idx] = amplitude_B  # Maintain amplitude until signals disappear
    
    # After both signals disappear, ensure they return to zero
    idx_end = np.where(t >= end_time_occurrence)[0]
    if idx_end.size > 0:
        signal_A[idx_end] = 0
        signal_B[idx_end] = 0

# Calculate cross-correlation
cross_corr = np.correlate(signal_A, signal_B, mode='full')
lags = np.arange(-len(signal_A) + 1, len(signal_A)) * dt

# Find the maximum correlation and corresponding lag
max_corr = np.max(cross_corr)
max_corr_lag = lags[np.argmax(cross_corr)]

# Plot the signals
plt.figure(figsize=(14, 10))

plt.subplot(3, 1, 1)
plt.plot(t, signal_A, label='Signal A')
plt.title('Signal A')
plt.ylabel('Amplitude')
plt.grid(True)
plt.legend()

plt.subplot(3, 1, 2)
plt.plot(t, signal_B, label='Signal B', color='orange')
plt.title('Signal B')
plt.ylabel('Amplitude')
plt.grid(True)
plt.legend()

# Plot cross-correlation
plt.subplot(3, 1, 3)
plt.plot(lags, cross_corr)
plt.title('Cross-Correlation between Signal A and Signal B')
plt.xlabel('Lag (s)')
plt.ylabel('Correlation')
plt.grid(True)

# Indicate the maximum correlation point
plt.axvline(x=max_corr_lag, color='red', linestyle='--', label=f'Max at lag {max_corr_lag:.2f}s')
plt.legend()

plt.tight_layout()
plt.show()

# Print the maximum correlation and corresponding lag
print(f"Maximum cross-correlation value: {max_corr}")
print(f"Lag at maximum cross-correlation: {max_corr_lag} seconds")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# Parameters for the First Plot
# -----------------------------
t_ramp = 2        # Duration of the ramp-up in seconds
t_hold = 20       # Duration to hold the peak value (10) in seconds
sampling_rate = 2 # Samples per second for smoothness

# ------------------------------
# Parameters for the Second Plot
# ------------------------------
x = 6             # Number of steps for the second plot
m = 2             # Time in seconds between each step

# --------------------------
# Define Delay 'n' Dynamically
# --------------------------
n = 1 #t_ramp        # Delay set to the time when the first plot reaches its maximum

# Validate that the second plot's steps fit within the hold time
if x * m > t_hold:
    raise ValueError("The total time for the second plot's steps (x * m) exceeds the hold time of the first plot (t_hold). "
                     "Please adjust 'x' or 'm' accordingly.")

# --------------------------
# Generate Time Array
# --------------------------
total_simulation_time = t_ramp + t_hold  # Total simulation time before drop
num_samples = int(total_simulation_time * sampling_rate) + 1  # +1 to include the endpoint
total_time = np.linspace(0, total_simulation_time, num_samples)

# --------------------------
# Generate First Plot Values
# --------------------------
# Ramp-up from 0 to 10
ramp_up_samples = int(t_ramp * sampling_rate)
ramp_up = np.linspace(0, 10, ramp_up_samples, endpoint=False)

# Hold at 10
hold_samples = int(t_hold * sampling_rate)
hold = np.ones(hold_samples) * 10

# Abrupt drop to 0
abrupt_drop = np.array([0])

# Combine values
values_1 = np.concatenate([ramp_up, hold, abrupt_drop])

# Ensure the length matches the total_time
if len(values_1) > len(total_time):
    values_1 = values_1[:len(total_time)]
elif len(values_1) < len(total_time):
    values_1 = np.pad(values_1, (0, len(total_time) - len(values_1)), 'constant')

# ---------------------------
# Generate Second Plot Values
# ---------------------------
# Initialize second plot values to zero
values_2 = np.zeros_like(total_time)

# Calculate step size
step_size = 10 / x

# Define step times starting after delay 'n'
step_times = n + m * np.arange(1, x + 1)  # e.g., [n + m, n + 2m, ..., n + xm]

# Apply each step
for step_time in step_times:
    # Find the index where the current step starts
    idx = np.searchsorted(total_time, step_time)
    if idx < len(values_2):
        values_2[idx:] += step_size
        # Prevent overshooting 10
        values_2 = np.clip(values_2, 0, 10)

# Abruptly drop to zero at the end
values_2[-1] = 0

# ---------------------------
# Plotting Both Plots
# ---------------------------
plt.figure(figsize=(12, 6))

# Plot First Signal
plt.plot(total_time, values_1, marker="o", label="First Plot (Ramp-Up & Hold)", linewidth=2)

# Plot Second Signal
plt.plot(total_time, values_2, marker="s", label="Second Plot (Stepped Increase)", linewidth=2, linestyle='--')

# Adding Legends and Labels
plt.xlabel("Time (s)", fontsize=14)
plt.ylabel("Value", fontsize=14)
#plt.title("Combined Plot: Ramp-Up & Stepped Increase with Synchronized Drop", fontsize=16)
plt.legend(fontsize=12)
plt.grid(True)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

# Display the Plot
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# Parameters for the First Plot
# -----------------------------
t_ramp = 1        # Duration of the ramp-up in seconds
t_hold = 100       # Duration to hold the peak value (10) in seconds
sampling_rate = 10 # Samples per second for smoothness

# ------------------------------
# Parameters for the Second Plot
# ------------------------------
x = 6             # Number of steps for the second plot
m = 1             # Time in seconds between each step

# --------------------------
# Define Delay 'n' Dynamically
# --------------------------
n = 5 #t_ramp        # Delay set to the time when the first plot reaches its maximum

# Validate that the second plot's steps fit within the hold time
if x * m > t_hold:
    raise ValueError("The total time for the second plot's steps (x * m) exceeds the hold time of the first plot (t_hold). "
                     "Please adjust 'x' or 'm' accordingly.")

# --------------------------
# Generate Time Array
# --------------------------
total_simulation_time = t_ramp + t_hold  # Total simulation time before drop
num_samples = int(total_simulation_time * sampling_rate) + 1  # +1 to include the endpoint
total_time = np.linspace(0, total_simulation_time, num_samples)

# --------------------------
# Generate First Plot Values
# --------------------------
# Ramp-up from 0 to 10
ramp_up_samples = int(t_ramp * sampling_rate)
ramp_up = np.linspace(0, 10, ramp_up_samples, endpoint=False)

# Hold at 10
hold_samples = int(t_hold * sampling_rate)
hold = np.ones(hold_samples) * 10

# Abrupt drop to 0
abrupt_drop = np.array([0])

# Combine values
values_1 = np.concatenate([ramp_up, hold, abrupt_drop])

# Ensure the length matches the total_time
if len(values_1) > len(total_time):
    values_1 = values_1[:len(total_time)]
elif len(values_1) < len(total_time):
    values_1 = np.pad(values_1, (0, len(total_time) - len(values_1)), 'constant')

# ---------------------------
# Generate Second Plot Values
# ---------------------------
# Initialize second plot values to zero
values_2 = np.zeros_like(total_time)

# Calculate step size
step_size = 10 / x

# Define step times starting after delay 'n'
step_times = n + m * np.arange(1, x + 1)  # e.g., [n + m, n + 2m, ..., n + xm]

# Apply each step
for step_time in step_times:
    # Find the index where the current step starts
    idx = np.searchsorted(total_time, step_time)
    if idx < len(values_2):
        values_2[idx:] += step_size
        # Prevent overshooting 10
        values_2 = np.clip(values_2, 0, 10)

# Abruptly drop to zero at the end
values_2[-1] = 0


# adding 10 seconds of zeros before the ramp up and after the abrupt drop
added_values = 50
# create a random number between 50 and 100

for i in range (5):
    added_values = np.random.randint(50,500)
    values_1 = np.concatenate([np.zeros(added_values),values_1,np.zeros(added_values)])
    values_2 = np.concatenate([np.zeros(added_values),values_2,np.zeros(added_values)])
    # dupliocte the values to simulate two signals
    values_1 = np.concatenate([values_1,values_1])
    values_2 = np.concatenate([values_2,values_2])

# ---------------------------
# Plotting Both Plots
# ---------------------------
plt.figure(figsize=(12, 6))

# Plot First Signal
plt.plot( values_1, marker="o", label="First Plot (Ramp-Up & Hold)", linewidth=2)

# Plot Second Signal
plt.plot( values_2, marker="s", label="Second Plot (Stepped Increase)", linewidth=2, linestyle='--')

# Adding Legends and Labels
plt.xlabel("Time (s)", fontsize=14)
plt.ylabel("Value", fontsize=14)
plt.title("Combined Plot: Ramp-Up & Stepped Increase with Synchronized Drop", fontsize=16)
plt.legend(fontsize=12)
plt.grid(True)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

# Display the Plot
plt.tight_layout()
plt.show()

# -----------------------------------
# Calculate and Plot Cross-Correlation
# -----------------------------------

# Import additional library for cross-correlation
from scipy.signal import correlate

# Calculate cross-correlation
cross_corr = correlate(values_2, values_1, mode='full')  # Full cross-correlation
lags = np.arange(-len(values_1) + 1, len(values_2))

# Normalize cross-correlation
cross_corr_normalized = cross_corr / (np.linalg.norm(values_1) * np.linalg.norm(values_2))

# Find the lag with the maximum cross-correlation
max_corr_index = np.argmax(cross_corr_normalized)
max_corr_lag = lags[max_corr_index]
max_corr_time = max_corr_lag / sampling_rate  # Convert lag to time

# Plot Cross-Correlation
plt.figure(figsize=(12, 6))
plt.plot(lags / sampling_rate, cross_corr_normalized)
plt.xlabel("Lag (s)", fontsize=14)
plt.ylabel("Normalized Cross-Correlation", fontsize=14)
plt.title("Cross-Correlation between First and Second Signals", fontsize=16)
plt.grid(True)
plt.axvline(x=max_corr_time, color='r', linestyle='--', label=f'Max Correlation at {max_corr_time} s')
plt.legend(fontsize=12)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.tight_layout()
plt.show()

# Print the lag with maximum cross-correlation
print(f"The maximum cross-correlation occurs at a lag of {max_corr_time} seconds.")
